In [1]:
# Load environment variables (ANTHROPIC_API_KEY) from .env
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Imports and shared classes

import json
import concurrent.futures
import re
from statistics import mean
from typing import Any, Callable, TypedDict, TypeVar

from utils.chat import Chat


class TestCase(TypedDict):
    task_description: str
    scenario: str
    prompt_inputs: dict[str, Any]
    solution_criteria: list[str]


class Grade(TypedDict):
    strengths: list[str]
    weaknesses: list[str]
    reasoning: str
    score: int


class EvalResult(TypedDict):
    output: str
    test_case: TestCase
    score: int
    reasoning: str

In [9]:
model = "claude-haiku-4-5"

In [3]:
# Report Builder
_REPORT_CSS = """
    body {
        font-family: Arial, sans-serif;
        line-height: 1.6;
        margin: 0;
        padding: 20px;
        color: #333;
    }
    .header {
        background-color: #f0f0f0;
        padding: 20px;
        border-radius: 5px;
        margin-bottom: 20px;
    }
    .summary-stats {
        display: flex;
        justify-content: space-between;
        flex-wrap: wrap;
        gap: 10px;
    }
    .stat-box {
        background-color: #fff;
        border-radius: 5px;
        padding: 15px;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        flex-basis: 30%;
        min-width: 200px;
    }
    .stat-value {
        font-size: 24px;
        font-weight: bold;
        margin-top: 5px;
    }
    table {
        width: 100%;
        border-collapse: collapse;
        margin-top: 20px;
    }
    th {
        background-color: #4a4a4a;
        color: white;
        text-align: left;
        padding: 12px;
    }
    td {
        padding: 10px;
        border-bottom: 1px solid #ddd;
        vertical-align: top;
    }
    tr:nth-child(even) {
        background-color: #f9f9f9;
    }
    .output-cell {
        white-space: pre-wrap;
    }
    .score {
        font-weight: bold;
        padding: 5px 10px;
        border-radius: 3px;
        display: inline-block;
    }
    .score-high {
        background-color: #c8e6c9;
        color: #2e7d32;
    }
    .score-medium {
        background-color: #fff9c4;
        color: #f57f17;
    }
    .score-low {
        background-color: #ffcdd2;
        color: #c62828;
    }
    .output {
        overflow: auto;
        white-space: pre-wrap;
    }
    .output pre {
        background-color: #f5f5f5;
        border: 1px solid #ddd;
        border-radius: 4px;
        padding: 10px;
        margin: 0;
        font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
        font-size: 14px;
        line-height: 1.4;
        color: #333;
        box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
        overflow-x: auto;
        white-space: pre-wrap;
        word-wrap: break-word;
    }
    td {
        width: 20%;
    }
    .score-col {
        width: 80px;
    }
"""

_MAX_SCORE = 10
_PASS_THRESHOLD = 7
_SCORE_HIGH_MIN = 8
_SCORE_LOW_MAX = 5


def _summarize(evaluation_results: list[EvalResult]) -> tuple[int, float, float]:
    total = len(evaluation_results)

    scores = [result["score"] for result in evaluation_results]
    avg = mean(scores) if scores else 0

    passed = len([s for s in scores if s >= _PASS_THRESHOLD])
    pass_rate = 100 * passed / total if total else 0

    return total, avg, pass_rate


def _score_class(score: int) -> str:
    if score >= _SCORE_HIGH_MIN:
        return "score-high"
    if score <= _SCORE_LOW_MAX:
        return "score-low"
    return "score-medium"


def _render_row(result: EvalResult) -> str:
    test_case = result["test_case"]

    prompt_inputs_html = "<br>".join(
        f"<strong>{key}:</strong> {value}"
        for key, value in test_case["prompt_inputs"].items()
    )
    criteria_string = "<br>• ".join(test_case["solution_criteria"])
    score = result["score"]

    return f"""
            <tr>
                <td>{test_case["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {_score_class(score)}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """


def generate_prompt_evaluation_report(evaluation_results: list[EvalResult]) -> str:
    total, avg, pass_rate = _summarize(evaluation_results)
    rows = "".join(_render_row(result) for result in evaluation_results)

    return f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>{_REPORT_CSS}</style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg:.1f} / {_MAX_SCORE}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥{_PASS_THRESHOLD})</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>{rows}
            </tbody>
        </table>
    </body>
    </html>
    """

In [4]:
# Prompt templates and shared utilities
_IDEAS_PROMPT = """
Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:

<task_description>
{task_description}
</task_description>

The prompt will receive the following inputs
<prompt_inputs>
{prompt_inputs_spec}
</prompt_inputs>

Each idea should represent a distinct scenario or example that tests different aspects of the task.

Output Format:
Provide your response as a structured JSON array where each item is a brief description of the idea.

Example:
```json
[
    "Testing with technical computer science terminology",
    "Testing with medical research findings",
    "Testing with complex mathematical concepts",
    ...
]
```

Ensure each idea is:
- Clearly distinct from the others
- Relevant to the task description
- Specific enough to guide generation of a full test case
- Quick to solve without requiring extensive computation or multi-step processing
- Solvable with no more than 400 tokens of output

Remember, only generate {num_cases} unique ideas
"""

_TEST_CASE_PROMPT = """
Generate a single detailed test case for a prompt evaluation based on:

<task_description>
{task_description}
</task_description>

<specific_idea>
{idea}
</specific_idea>

<allowed_input_keys>
{allowed_keys}
</allowed_input_keys>

Output Format:
```json
{{
    "prompt_inputs": {{
    {example_prompt_inputs}
    }},
    "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
}}
```

IMPORTANT REQUIREMENTS:
- You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}
- Do NOT add any additional keys to prompt_inputs
- All keys listed in allowed_input_keys must be included in your response
- Make the test case realistic and practically useful
- Include measurable, concise solution criteria
- The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
- Avoid over-specifying criteria with requirements that go beyond the core task
- Keep solution criteria simple, focused, and directly tied to the fundamental task
- The test case should be tailored to the specific idea provided
- Quick to solve without requiring extensive computation or multi-step processing
- Solvable with no more than 400 tokens of output
- DO NOT include any fields beyond those specified in the output format

Here's an example of a sample input with an ideal output:
<sample_input>
<sample_task_description>
Extract topics out of a passage of text
</sample_task_description>
<sample_specific_idea>
Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
</sample_specific_idea>

<sample_allowed_input_keys>
"content"
</sample_allowed_input_keys>
</sample_input>
<ideal_output>
```json
{
    "prompt_inputs": {
        "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
    },
    "solution_criteria": [
        "Includes all topics mentioned"
    ]
}
```
</ideal_output>
This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
"""

_EXTRA_CRITERIA_PROMPT = """
Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
<extra_important_criteria>
{extra_criteria}
</extra_important_criteria>
"""

_GRADE_PROMPT = """
Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

Original task description:
<task_description>
{task_description}
</task_description>

Original task inputs:
<task_inputs>
{{ {prompt_inputs} }}
</task_inputs>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{solution_criteria}
</criteria>

{extra_criteria_section}

Scoring Guidelines:
* Score 1-3: Solution fails to meet one or more MANDATORY requirements
* Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
* Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
* Score 9-10: Solution meets all mandatory and secondary criteria

IMPORTANT SCORING INSTRUCTIONS:
* Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
* If a solution meets all of the mandatory and secondary criteria give it a 10
* Don't complain that the solution "only" meets the mandatory and secondary criteria. Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
* ANY violation of a mandatory requirement MUST result in a score of 3 or lower
* The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

_T = TypeVar("_T")
_R = TypeVar("_R")


def render(template_string: str, variables: dict[str, Any]) -> str:
    placeholders = re.findall(r"{([^{}]+)}", template_string)

    result = template_string
    for placeholder in placeholders:
        if placeholder in variables:
            result = result.replace(
                "{" + placeholder + "}", str(variables[placeholder])
            )

    return result.replace("{{", "{").replace("}}", "}")


def _run_concurrently(
    items: list[_T],
    worker: Callable[[_T], _R],
    max_workers: int,
    label: str,
    skip_errors: bool = False,
) -> list[_R]:
    """Run worker(item) across a thread pool, printing progress at 20% milestones."""
    results: list[_R] = []
    total = len(items)
    completed = 0
    last_reported = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(worker, item) for item in items]

        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
            except Exception as e:
                if not skip_errors:
                    raise
                print(f"Error: {e}")
                continue

            completed += 1
            milestone = (int(completed / total * 100) // 20) * 20
            if milestone > last_reported:
                print(f"{label} {completed}/{total} test cases")
                last_reported = milestone
            results.append(result)

    return results

In [5]:
class DatasetGenerator:
    def __init__(self, max_concurrent_tasks: int = 3) -> None:
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_unique_ideas(
        self,
        task_description: str,
        prompt_inputs_spec: dict[str, Any],
        num_cases: int,
    ) -> list[str]:
        """Generate a list of unique ideas for test cases based on the task description"""
        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = str(value).replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = render(
            _IDEAS_PROMPT,
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        text = (
            Chat(model=model, system=system_prompt, temperature=1.0)
            .user(rendered_prompt)
            .send(prefill="```json", stop_sequences=["```"])
        )

        return json.loads(text)

    def generate_test_case(
        self,
        task_description: str,
        idea: str,
        prompt_inputs_spec: dict[str, Any] = {},
    ) -> TestCase:
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = str(value).replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = render(
            _TEST_CASE_PROMPT,
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        text = (
            Chat(model=model, system=system_prompt, temperature=0.7)
            .user(rendered_prompt)
            .send(prefill="```json", stop_sequences=["```"])
        )

        test_case: TestCase = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description: str,
        prompt_inputs_spec: dict[str, Any] = {},
        num_cases: int = 1,
        output_file: str = "dataset.json",
    ) -> list[TestCase]:
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = _run_concurrently(
            ideas,
            lambda idea: self.generate_test_case(
                task_description, idea, prompt_inputs_spec
            ),
            self.max_concurrent_tasks,
            "Generated",
            skip_errors=True,
        )

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

In [6]:
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks: int = 3) -> None:
        self.max_concurrent_tasks = max_concurrent_tasks

    def grade_output(
        self,
        test_case: TestCase,
        output: str,
        extra_criteria: str | None,
    ) -> Grade:
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = str(value).replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_section = render(
                _EXTRA_CRITERIA_PROMPT,
                {"extra_criteria": extra_criteria},
            )

        eval_prompt = render(
            _GRADE_PROMPT,
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        eval_text = (
            Chat(model=model, temperature=0.0)
            .user(eval_prompt)
            .send(prefill="```json", stop_sequences=["```"])
        )
        return json.loads(eval_text)

    def run_test_case(
        self,
        test_case: TestCase,
        run_prompt_function: Callable[[dict[str, Any]], str],
        extra_criteria: str | None = None,
    ) -> EvalResult:
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function: Callable[[dict[str, Any]], str],
        dataset_file: str,
        extra_criteria: str | None = None,
        json_output_file: str = "output.json",
        html_output_file: str = "output.html",
    ) -> list[EvalResult]:
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = _run_concurrently(
            dataset,
            lambda test_case: self.run_test_case(
                test_case, run_prompt_function, extra_criteria
            ),
            self.max_concurrent_tasks,
            "Graded",
        )

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [7]:
# Create the dataset generator and evaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
generator = DatasetGenerator(max_concurrent_tasks=1)
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [10]:
dataset = generator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Write a compact, concise 1 day meal plan for a single athlete.",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Athlete's goal",
        "restrictions": "Athlete's dietary restrictions",
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [ ]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case

# How to improve prompts:
# 1. be clear and direct
# 2. be specific (e.g. add guidelines or steps that the model should follow)

#    Guidelines:
#    1. Include accurate daily calorie amount
#    2. Show protein, fat and carb amounts
#    3. Specify when to eat each meal
#    4. Use only foods that fit restrictions
#    5. List all portion sizes in grams
#    6. Keep budget-friendly if mentioned

#    Follow these steps:
#    1. Calculate daily calories needed
#    2. Figure out protein, fat, carb amounts
#    3. Plan meal timing around workouts
#    4. Choose foods that fit restrictions
#    5. Set protein sizes in grams
#    6. Adjust for budget if needed

# 3. provide structure in prompts by using XML tags
# 4. provide examples (few-shot learning/multi-shot prompting)



def run_prompt(prompt_inputs: dict[str, Any]) -> str:
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietaryrestrictions.
    
    <athlete_information>
    - Height: {prompt_inputs["height"]}
    - Weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions: {prompt_inputs["restrictions"]}
    </athlete_information>

    Guidelines:
    1. Include accurate daily calorie amount
    2. Show protein, fat and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned
    """

    return Chat(model=model).user(prompt).send()

In [22]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions and timings
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.333333333333333
